In [2]:
# If this is on Github or Kaggle then pls change the path accordingly. The current paths are for local machine.
# Also create a TrainingTrial folder inside Models and place the model_alg_predictor.ipynb file there.
# The dataset should be placed in the datasets folder as mentioned in the code.
# Or you Can just change the paths in the code to match your local setup. 
# The code is designed to be run as a Jupyter Notebook.
# dataset should be taken as ml_projects_dataset.csv and placed in the datasets folder.

# IMPORTS


import pandas as pd
import numpy as np
import sqlite3
import optuna
import pickle

from datetime import datetime

from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split

from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier

from sklearn.metrics import accuracy_score

# PATHS


dataset_path = (
    r"C:\Users\Uday\Desktop\Coding\Language\Python\ML\ml_env\Work\datasets\ml_projects_dataset.csv"
)

optuna_db = (
    r"C:\Users\Uday\Desktop\Coding\Language\Python\ML\ml_env\Work\Models\TrainingTrial\trials_ml_alg_predictor.db"
)

app_db = (
    r"C:\Users\Uday\Desktop\Coding\Language\Python\ML\ml_env\Work\Models\TrainingTrial\ml_predictor.db"
)

rf_model_path = (
    r"C:\Users\Uday\Desktop\Coding\Language\Python\ML\ml_env\Work\Models\TrainingTrial\random_forest_model.pkl"
)

dt_model_path = (
    r"C:\Users\Uday\Desktop\Coding\Language\Python\ML\ml_env\Work\Models\TrainingTrial\decision_tree_model.pkl"
)

feature_encoder_path = (
    r"C:\Users\Uday\Desktop\Coding\Language\Python\ML\ml_env\Work\Models\TrainingTrial\feature_encoders.pkl"
)

target_encoder_path = (
    r"C:\Users\Uday\Desktop\Coding\Language\Python\ML\ml_env\Work\Models\TrainingTrial\target_encoder.pkl"
)

output_path = (
    r"C:\Users\Uday\Desktop\Coding\Language\Python\ML\ml_env\Work\Models\TrainingTrial\output.txt"
)


# LOAD DATASET


df = pd.read_csv(dataset_path)


# CLEAN DATASET


for column in df.columns:

    df[column] = df[column].astype(str)

    df[column] = df[column].str.strip()

    df[column] = df[column].str.lower()

# REMOVE NULLS

df = df.dropna()

# REMOVE DUPLICATES

df = df.drop_duplicates(
    ignore_index=True
)


# DATASET INFORMATION


print("\nDataset Shape:")
print(df.shape)

print("\nColumns:")
print(df.columns)

print("\nFirst 5 Rows:")
print(df.head())

# ALLOWED VALUES


print("\n==============================")
print("ALLOWED VALUES FROM DATASET")
print("==============================")

for column in df.columns:

    print(f"\n{column}:")

    print(
        sorted(
            df[column].unique()
        )
    )
# FEATURES AND TARGET


X = df.drop(
    "model used",
    axis=1
).copy()

y = df["model used"]

# FEATURE ENCODERS


encoders = {}

for column in X.columns:

    encoder = LabelEncoder()

    X[column] = encoder.fit_transform(
        X[column]
    )

    encoders[column] = encoder

# SAVE FEATURE ENCODERS

with open(
    feature_encoder_path,
    "wb"
) as file:

    pickle.dump(
        encoders,
        file
    )


# TARGET ENCODER


target_encoder = LabelEncoder()

y = target_encoder.fit_transform(
    y
)

# SAVE TARGET ENCODER

with open(
    target_encoder_path,
    "wb"
) as file:

    pickle.dump(
        target_encoder,
        file
    )

# TRAIN TEST SPLIT


X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("\nTraining Samples:")
print(len(X_train))

print("\nTesting Samples:")
print(len(X_test))

print("\nNumber Of Algorithms:")
print(len(target_encoder.classes_))

print("\nAlgorithms In Dataset:")
print(target_encoder.classes_)

# RANDOM FOREST OBJECTIVE


def RF_objective(trial):

    params = {

        "n_estimators": trial.suggest_int(
            "n_estimators",
            50,
            300
        ),

        "max_depth": trial.suggest_int(
            "max_depth",
            2,
            32
        ),

        "min_samples_split": trial.suggest_int(
            "min_samples_split",
            2,
            20
        ),

        "min_samples_leaf": trial.suggest_int(
            "min_samples_leaf",
            1,
            20
        ),

        "random_state": 42,
        "n_jobs": -1
    }

    model = RandomForestClassifier(
        **params
    )

    model.fit(
        X_train,
        y_train
    )

    predictions = model.predict(
        X_test
    )

    accuracy = accuracy_score(
        y_test,
        predictions
    )

    return accuracy

# RANDOM FOREST STUDY


rf_study = optuna.create_study(
    study_name="random_forest_study",
    storage=f"sqlite:///{optuna_db}",
    load_if_exists=True,
    direction="maximize"
)

rf_study.optimize(
    RF_objective,
    n_trials=50
)

rf_model = RandomForestClassifier(
    **rf_study.best_params,
    random_state=42,
    n_jobs=-1
)

rf_model.fit(
    X_train,
    y_train
)

predictions_rf = rf_model.predict(
    X_test
)

accuracy_rf = accuracy_score(
    y_test,
    predictions_rf
)

print("\nRandom Forest Accuracy:")
print(accuracy_rf)

print("\nRandom Forest Best Parameters:")
print(rf_study.best_params)

# SAVE RANDOM FOREST MODEL

with open(
    rf_model_path,
    "wb"
) as file:

    pickle.dump(
        rf_model,
        file
    )


# DECISION TREE OBJECTIVE


def DT_objective(trial):

    params = {

        "max_depth": trial.suggest_int(
            "max_depth",
            2,
            32
        ),

        "min_samples_split": trial.suggest_int(
            "min_samples_split",
            2,
            20
        ),

        "min_samples_leaf": trial.suggest_int(
            "min_samples_leaf",
            1,
            20
        ),

        "random_state": 42
    }

    model = DecisionTreeClassifier(
        **params
    )

    model.fit(
        X_train,
        y_train
    )

    predictions = model.predict(
        X_test
    )

    accuracy = accuracy_score(
        y_test,
        predictions
    )

    return accuracy


# DECISION TREE STUDY


dt_study = optuna.create_study(
    study_name="decision_tree_study",
    storage=f"sqlite:///{optuna_db}",
    load_if_exists=True,
    direction="maximize"
)

dt_study.optimize(
    DT_objective,
    n_trials=50
)

dt_model = DecisionTreeClassifier(
    **dt_study.best_params,
    random_state=42
)

dt_model.fit(
    X_train,
    y_train
)

predictions_dt = dt_model.predict(
    X_test
)

accuracy_dt = accuracy_score(
    y_test,
    predictions_dt
)

print("\nDecision Tree Accuracy:")
print(accuracy_dt)

print("\nDecision Tree Best Parameters:")
print(dt_study.best_params)


with open(
    dt_model_path,
    "wb"
) as file:

    pickle.dump(
        dt_model,
        file
    )
# TRAINING HISTORY


connection = sqlite3.connect(
    app_db
)

cursor = connection.cursor()

cursor.execute("""

CREATE TABLE IF NOT EXISTS training_history (

    id INTEGER PRIMARY KEY AUTOINCREMENT,

    timestamp TEXT,

    rf_accuracy REAL,

    dt_accuracy REAL,

    best_model TEXT

)

""")

if accuracy_rf >= accuracy_dt:

    best_model_name = "random forest"

else:

    best_model_name = "decision tree"

cursor.execute("""

INSERT INTO training_history (

    timestamp,
    rf_accuracy,
    dt_accuracy,
    best_model

)

VALUES (?, ?, ?, ?)

""", (

    str(datetime.now()),
    accuracy_rf,
    accuracy_dt,
    best_model_name

))

connection.commit()

print("\nTRAINING HISTORY")

history_df = pd.read_sql_query(
    """

    SELECT *

    FROM training_history

    ORDER BY id DESC

    LIMIT 10

    """,
    connection
)

print(history_df)

connection.close()


# BEST MODEL SELECTION


if accuracy_rf >= accuracy_dt:

    print("\nRandom Forest is Best")

    full_model = rf_model

else:

    print("\nDecision Tree is Best")

    full_model = dt_model

# SAVE BEST MODEL

with open(
    r"C:\Users\Uday\Desktop\Coding\Language\Python\ML\ml_env\Work\Models\TrainingTrial\best_model.pkl",
    "wb"
) as file:

    pickle.dump(
        full_model,
        file
    )

# OUTPUT FILE


with open(
    output_path,
    "w",
    encoding="utf-8"
) as file:

    file.write(
        f"Random Forest Accuracy: {accuracy_rf}\n"
    )

    file.write(
        f"Decision Tree Accuracy: {accuracy_dt}\n"
    )

    file.write(
        f"Best Model: {best_model_name}\n"
    )

# USER INPUT


print("\nENTER PROJECT DETAILS")

user_data = {}

for column in X.columns:

    print(f"\nAllowed values for {column}:")
    print(encoders[column].classes_)

    value = input(f"{column}: ").strip().lower()

    # VALIDATION

    if value not in encoders[column].classes_:

        print(f"\nInvalid value for {column}")

        print("Allowed values are:")

        print(encoders[column].classes_)

        exit()

    # ENCODE

    encoded_value = encoders[column].transform(
        [value]
    )[0]

    user_data[column] = encoded_value


# PREDICTION


user_df = pd.DataFrame([user_data])

prediction_encoded = full_model.predict(
    user_df
)

prediction = target_encoder.inverse_transform(
    prediction_encoded
)

probabilities = full_model.predict_proba(
    user_df
)

confidence = np.max(probabilities) * 100

print("\nRECOMMENDED ALGORITHM:")

print(prediction[0])

print(f"\nConfidence: {confidence:.2f}%")



# FEEDBACK SECTION


print("\nFEEDBACK SECTION")

feedback_choice = input(
    "Do you want to give feedback? (yes/no): "
).strip().lower()

if feedback_choice == "yes":

    correct_answer = input(
        "\nWhat was the correct algorithm?: "
    ).strip().lower()

    if correct_answer not in target_encoder.classes_:

        print("\nInvalid Algorithm Name")

    else:

        user_inputs = {}

        for column in X.columns:

            decoded_value = encoders[column].inverse_transform(
                [user_data[column]]
            )[0]

            user_inputs[column] = decoded_value

        connection = sqlite3.connect(app_db)

        cursor = connection.cursor()

        cursor.execute("""

        CREATE TABLE IF NOT EXISTS feedback (

            id INTEGER PRIMARY KEY AUTOINCREMENT,

            timestamp TEXT,

            predicted_algorithm TEXT,

            correct_algorithm TEXT,

            confidence REAL,

            learning TEXT,

            input TEXT,

            output TEXT,

            task_type TEXT,

            sequential_data TEXT,

            dataset_size TEXT,

            structured_data TEXT,

            real_time_prediction TEXT,

            interpretability_needed TEXT,

            noise_level TEXT,

            accuracy_priority TEXT,

            gpu_available TEXT,

            linear_relationship TEXT,

            num_classes TEXT,

            feature_dimension TEXT

        )

        """)

        cursor.execute("""

        INSERT INTO feedback (

            timestamp,
            predicted_algorithm,
            correct_algorithm,
            confidence,
            learning,
            input,
            output,
            task_type,
            sequential_data,
            dataset_size,
            structured_data,
            real_time_prediction,
            interpretability_needed,
            noise_level,
            accuracy_priority,
            gpu_available,
            linear_relationship,
            num_classes,
            feature_dimension

        )

        VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)

        """, (

            str(datetime.now()),
            prediction[0],
            correct_answer,
            confidence,
            user_inputs["learning"],
            user_inputs["input"],
            user_inputs["output"],
            user_inputs["task_type"],
            user_inputs["sequential_data"],
            user_inputs["dataset_size"],
            user_inputs["structured_data"],
            user_inputs["real_time_prediction"],
            user_inputs["interpretability_needed"],
            user_inputs["noise_level"],
            user_inputs["accuracy_priority"],
            user_inputs["gpu_available"],
            user_inputs["linear_relationship"],
            user_inputs["num_classes"],
            user_inputs["feature_dimension"]

        ))

        connection.commit()

        # SAVE FEEDBACK TO MAIN DATASET

        feedback_df = pd.DataFrame([{

            "learning": user_inputs["learning"],
            "input": user_inputs["input"],
            "output": user_inputs["output"],
            "task_type": user_inputs["task_type"],
            "sequential_data": user_inputs["sequential_data"],
            "dataset_size": user_inputs["dataset_size"],
            "structured_data": user_inputs["structured_data"],
            "real_time_prediction": user_inputs["real_time_prediction"],
            "interpretability_needed": user_inputs["interpretability_needed"],
            "noise_level": user_inputs["noise_level"],
            "accuracy_priority": user_inputs["accuracy_priority"],
            "gpu_available": user_inputs["gpu_available"],
            "linear_relationship": user_inputs["linear_relationship"],
            "num_classes": user_inputs["num_classes"],
            "feature_dimension": user_inputs["feature_dimension"],
            "model used": correct_answer

        }])

        feedback_df = feedback_df[df.columns]

        updated_main_df = pd.concat(
            [df, feedback_df],
            ignore_index=True
        )

        updated_main_df = updated_main_df.drop_duplicates()

        updated_main_df.to_csv(
            r"C:\Users\Uday\Desktop\Coding\Language\Python\ML\ml_env\Work\datasets\ml_projects_dataset.csv",
            index=False
        )

        connection.close()


        print("\nFeedback Stored Successfully")


Dataset Shape:
(815, 16)

Columns:
Index(['learning', 'input', 'output', 'task_type', 'sequential_data',
       'dataset_size', 'structured_data', 'real_time_prediction',
       'interpretability_needed', 'noise_level', 'accuracy_priority',
       'gpu_available', 'linear_relationship', 'num_classes',
       'feature_dimension', 'model used'],
      dtype='object')

First 5 Rows:
     learning    input       output task_type sequential_data dataset_size  \
0  supervised     text         text  generate             yes        large   
1  supervised  tabular  probability      flag              no       medium   
2  supervised  tabular  probability      flag              no        large   
3  supervised  tabular       number   predict              no       medium   
4  supervised  tabular  probability  classify              no       medium   

  structured_data real_time_prediction interpretability_needed noise_level  \
0    unstructured                   no                      no       

[I 2026-07-21 17:24:46,218] Using an existing study with name 'random_forest_study' instead of creating a new one.
[I 2026-07-21 17:24:46,800] Trial 100 finished with value: 0.803680981595092 and parameters: {'n_estimators': 212, 'max_depth': 26, 'min_samples_split': 15, 'min_samples_leaf': 2}. Best is trial 5 with value: 0.8098159509202454.
[I 2026-07-21 17:24:47,399] Trial 101 finished with value: 0.8282208588957055 and parameters: {'n_estimators': 178, 'max_depth': 28, 'min_samples_split': 14, 'min_samples_leaf': 1}. Best is trial 101 with value: 0.8282208588957055.
[I 2026-07-21 17:24:47,958] Trial 102 finished with value: 0.8282208588957055 and parameters: {'n_estimators': 179, 'max_depth': 28, 'min_samples_split': 14, 'min_samples_leaf': 1}. Best is trial 101 with value: 0.8282208588957055.
[I 2026-07-21 17:24:48,643] Trial 103 finished with value: 0.8282208588957055 and parameters: {'n_estimators': 179, 'max_depth': 27, 'min_samples_split': 14, 'min_samples_leaf': 1}. Best is tr


Random Forest Accuracy:
0.8343558282208589

Random Forest Best Parameters:
{'n_estimators': 185, 'max_depth': 24, 'min_samples_split': 17, 'min_samples_leaf': 1}


[I 2026-07-21 17:25:34,840] Trial 101 finished with value: 0.7423312883435583 and parameters: {'max_depth': 19, 'min_samples_split': 13, 'min_samples_leaf': 9}. Best is trial 14 with value: 0.754601226993865.
[I 2026-07-21 17:25:34,934] Trial 102 finished with value: 0.7607361963190185 and parameters: {'max_depth': 21, 'min_samples_split': 15, 'min_samples_leaf': 8}. Best is trial 102 with value: 0.7607361963190185.
[I 2026-07-21 17:25:35,001] Trial 103 finished with value: 0.7361963190184049 and parameters: {'max_depth': 21, 'min_samples_split': 15, 'min_samples_leaf': 7}. Best is trial 102 with value: 0.7607361963190185.
[I 2026-07-21 17:25:35,064] Trial 104 finished with value: 0.7607361963190185 and parameters: {'max_depth': 22, 'min_samples_split': 16, 'min_samples_leaf': 8}. Best is trial 102 with value: 0.7607361963190185.
[I 2026-07-21 17:25:35,128] Trial 105 finished with value: 0.6871165644171779 and parameters: {'max_depth': 22, 'min_samples_split': 15, 'min_samples_leaf': 1


Decision Tree Accuracy:
0.7607361963190185

Decision Tree Best Parameters:
{'max_depth': 21, 'min_samples_split': 15, 'min_samples_leaf': 8}

TRAINING HISTORY
   id                   timestamp  rf_accuracy  dt_accuracy     best_model
0   2  2026-07-21 17:25:38.031218     0.834356     0.760736  random forest
1   1  2026-06-02 13:06:39.456849     0.809816     0.754601  random forest

Random Forest is Best

ENTER PROJECT DETAILS

Allowed values for learning:
['reinforcements' 'self-supervised' 'semi-supervised' 'supervised'
 'unsupervised']

Allowed values for input:
['audio' 'image' 'label' 'mixed' 'number' 'tabular' 'text' 'time_series'
 'video']

Allowed values for output:
['audio' 'image' 'label' 'mixed' 'number' 'probability' 'tabular' 'text'
 'time_series' 'video']

Allowed values for task_type:
['classify' 'detect' 'flag' 'forecast' 'generate' 'group' 'predict'
 'recommend' 'transcribe' 'translate']

Allowed values for sequential_data:
['no' 'yes']

Allowed values for dataset_size

In [ ]:
import base64

encoded_string = "bWFkZSBieSB2cnVzaGFiaC5zIHN3cw=="
print(base64.b64decode(encoded_string).decode('utf-8'))

